In [1]:
# =====================================================================
# CULTURAL CORPS INTEGRATION - STEP 4 OF 5: MERGE INTO MAIN PIPELINE
# =====================================================================
# Standalone - reads the cc_ files from Steps 2/3 plus the main pipeline
# files, and merges them together. After this runs, every existing
# downstream script (qcew_vs_internal_comparison.py, exploration_round2.py,
# raw_numbers_and_shares.py, historical_trend_raw_counts.py) picks up
# Cultural Corps automatically - none of them need to change, since they
# just read AllRoles_nioccs_ready.csv / AllRoles_nioccs_coded.csv, which
# this script updates in place.
#
# Backs up both main files before touching them, same safety pattern as
# the 2023 rerun merge-back earlier in this project.
#
# IMPORTANT: this does NOT regenerate AllRoles_nioccs_coded_with_education.csv.
# Cultural Corps rows haven't been through education-level tagging yet.
# After this script runs, re-run tag_education_requirements.py once more -
# it already reads from AllRoles_nioccs_coded.csv (which this script
# updates to include Cultural Corps), so re-running it picks up the new
# rows with no changes needed to that script either.

import pandas as pd
import shutil
import os

MAIN_READY_PATH = "../../data/AllRoles_nioccs_ready.csv"
MAIN_CODED_PATH = "../../data/AllRoles_nioccs_coded.csv"

CC_READY_PATH = "../../data/cultural_corps/cc_step2_ready_for_nioccs.csv"
CC_CODED_PATH = "../../data/cultural_corps/cc_step3_coded.csv"

BACKUP_READY_PATH = "../../data/AllRoles_nioccs_ready_BACKUP_before_cultural_corps.csv"
BACKUP_CODED_PATH = "../../data/AllRoles_nioccs_coded_BACKUP_before_cultural_corps.csv"


def merge_file_pair(main_path, cc_path, backup_path, label):
    print(f"\n--- Merging {label} ---")

    main_df = pd.read_csv(main_path, encoding="utf-8-sig")
    cc_df = pd.read_csv(cc_path, encoding="utf-8-sig")
    print(f"Main file rows: {len(main_df)}")
    print(f"Cultural Corps file rows: {len(cc_df)}")

    # Defensive check: these two ID namespaces should never overlap (core
    # hub Opportunity Ids are GUIDs from InPlace/Airtable; Cultural Corps
    # rows share that same source, but flag it explicitly rather than
    # silently creating duplicate rows if it ever does happen).
    overlap = set(main_df["Opportunity Identifier"]) & set(cc_df["Opportunity Identifier"])
    if overlap:
        print(f"WARNING: {len(overlap)} Opportunity Identifiers appear in BOTH files - "
              f"these will be deduplicated (main file version kept) rather than double-counted.")

    shutil.copy(main_path, backup_path)
    print(f"Backed up existing file to {backup_path}")

    combined = pd.concat([main_df, cc_df], ignore_index=True)
    combined = combined.drop_duplicates(subset=["Opportunity Identifier"], keep="first")

    combined.to_csv(main_path, index=False)
    print(f"Wrote {main_path}: {len(combined)} total rows "
          f"(was {len(main_df)}, added {len(combined) - len(main_df)})")

    return combined


def main():
    for path in [MAIN_READY_PATH, MAIN_CODED_PATH, CC_READY_PATH, CC_CODED_PATH]:
        if not os.path.exists(path):
            raise FileNotFoundError(
                f"{path} doesn't exist. Make sure Steps 1-3 have been run and produced "
                f"their expected output files before running this merge."
            )

    ready_combined = merge_file_pair(MAIN_READY_PATH, CC_READY_PATH, BACKUP_READY_PATH, "ready files")
    coded_combined = merge_file_pair(MAIN_CODED_PATH, CC_CODED_PATH, BACKUP_CODED_PATH, "coded files")

    print("\n" + "=" * 70)
    print("Post-merge summary")
    print("=" * 70)
    print("\nHub distribution in merged ready file:")
    print(ready_combined["Program Hub Category"].value_counts().to_string())

    print("\nCohort Year distribution in merged ready file:")
    print(ready_combined["Cohort Year"].value_counts(dropna=False).sort_index().to_string())

    n_coded_errors = coded_combined["API Call Error"].notna().sum()
    print(f"\nMerged coded file: {len(coded_combined)} rows, "
          f"{n_coded_errors} with an API error or intentional skip flag.")

    print("\n" + "=" * 70)
    print("NEXT STEP")
    print("=" * 70)
    print("Re-run tag_education_requirements.py now - it reads AllRoles_nioccs_coded.csv "
          "(just updated above) and will regenerate AllRoles_nioccs_coded_with_education.csv "
          "with Cultural Corps rows properly tagged. No changes needed to that script.")
    print("\nAfter that, Step 5: selectively update the deck's Arts and Rec slide and any "
          "appendix tables with the real merged numbers.")


if __name__ == "__main__":
    main()


--- Merging ready files ---
Main file rows: 7125
Cultural Corps file rows: 1457
Backed up existing file to ../../data/AllRoles_nioccs_ready_BACKUP_before_cultural_corps.csv
Wrote ../../data/AllRoles_nioccs_ready.csv: 8582 total rows (was 7125, added 1457)

--- Merging coded files ---
Main file rows: 7125
Cultural Corps file rows: 1457
Backed up existing file to ../../data/AllRoles_nioccs_coded_BACKUP_before_cultural_corps.csv
Wrote ../../data/AllRoles_nioccs_coded.csv: 8582 total rows (was 7125, added 1457)

Post-merge summary

Hub distribution in merged ready file:
Program Hub Category
Community and Social Services          2398
Marketing and Communications           1963
Arts, Entertainment, and Recreation    1457
STEM and Green                         1445
Healthcare                             1319

Cohort Year distribution in merged ready file:
Cohort Year
2022.0     812
2023.0    1681
2024.0    1772
2025.0    2013
2026.0    2293
NaN         11

Merged coded file: 8582 rows, 617 